### IMPORT THE PACKAGES

In [2]:
import os, re
import sys
import random
import numpy as np
import pandas as pd
from astropy.io import fits
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import matplotlib.dates as mdates
from datetime import timedelta

import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Reshape, LSTM, Dense
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import MeanSquaredError, MeanAbsoluteError
from sklearn.metrics import mean_absolute_error as MAE, mean_squared_error as MSE

from sklearn.model_selection import KFold
from scipy.stats import pearsonr
try:
    current_folder = os.getcwd()
    hsemetric_folder = os.path.abspath(os.path.join(current_folder, "../HSEMETRIC"))
    sys.path.append(hsemetric_folder)
    import hsemetric
except Exception:
    hsemetric = None
    print("hsemetric not available — Threat_score will be set to NaN.")

### CHECK GPU STATUS

In [4]:
gpus = tf.config.list_physical_devices('GPU')
print("GPUs:", gpus)

# Optional: more detail
for gpu in gpus:
    details = tf.config.experimental.get_device_details(gpu)
    print(details)

GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
{'compute_capability': (7, 5), 'device_name': 'Quadro RTX 5000'}
{'compute_capability': (7, 5), 'device_name': 'Quadro RTX 5000'}


2025-12-09 13:55:08.019567: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:961] could not open file to read NUMA node: /sys/bus/pci/devices/0000:17:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-12-09 13:55:08.020121: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:961] could not open file to read NUMA node: /sys/bus/pci/devices/0000:73:00.0/numa_node
Your kernel may have been built without NUMA support.


### ABLATION MODEL

In [5]:
# -------------------------
# SpeedNet Model definition
# -------------------------
def SpeedNet(input_shape, lstm_unit=30, dense_unit=100):
    tf.keras.backend.clear_session()
    random.seed(42)
    np.random.seed(42)
    tf.random.set_seed(42)

    cnn_input = Input(shape=input_shape)
    x = Conv2D(16, (3, 3), activation="relu")(cnn_input)
    x = MaxPooling2D((2, 2))(x)
    x = Conv2D(32, (3, 3), activation="relu")(x)
    x = MaxPooling2D((2, 2))(x)
    x = Conv2D(64, (3, 3), activation="relu")(x)
    x = MaxPooling2D((2, 2))(x)
    x = Conv2D(128, (3, 3), activation="relu")(x)
    x = MaxPooling2D((2, 2))(x)
    cnn_output = Flatten()(x)

    cnn_output_reshaped = Reshape((1, cnn_output.shape[1]))(cnn_output)
    lstm_output = LSTM(lstm_unit, activation="relu")(cnn_output_reshaped)
    dense_lay = Dense(dense_unit, activation="relu")(lstm_output)
    lstm_output = Dense(1, activation="linear")(dense_lay)

    combined_model = Model(inputs=cnn_input, outputs=lstm_output)
    return combined_model

# -------------------------
# Data preparation
# -------------------------
def prepare_dataset(sw_csv, icme_excel, img_path):
    df_sw = pd.read_csv(sw_csv)
    df_sw["DATE"] = pd.to_datetime(df_sw["DATE"])
    df_sw = df_sw.rename(columns={"V(Km/s)": "speed"})

    icme_list = pd.read_excel(icme_excel)
    icme_list["Start_time"] = pd.to_datetime(icme_list["Start_time"].str.split(" ").str[0], format="%Y/%m/%d")
    icme_list["End_time"] = pd.to_datetime(icme_list["End_time"].str.split(" ").str[0], format="%Y/%m/%d")

    icme_dates = []
    for start, end in zip(icme_list["Start_time"], icme_list["End_time"]):
        icme_dates.extend(pd.date_range(start, end).tolist())
    icme_dates = set(icme_dates)

    df_sw = df_sw[~df_sw["DATE"].isin(icme_dates)].reset_index(drop=True)

    img_files, img_dates = [], []
    for f in os.listdir(img_path):
        if f.isdigit() and len(f) == 8:
            date = pd.to_datetime(f, format="%Y%m%d")
        else:
            m = re.search(r"(\d{8})", f)
            if not m:
                continue
            date = pd.to_datetime(m.group(1), format="%Y%m%d")
        if date in icme_dates:
            continue
        img_files.append(f)
        img_dates.append(date)

    img_dates, img_files = zip(*sorted(zip(img_dates, img_files)))
    return df_sw, dict(zip(img_dates, img_files)), icme_dates

# -------------------------
# Match maps to SW
# -------------------------
def match_data(df_sw, img_dict, delay):
    matched = []
    for _, row in df_sw.iterrows():
        sw_date, sw_speed = row["DATE"], row["speed"]
        img_date = sw_date - timedelta(days=delay)
        if img_date in img_dict:
            matched.append((sw_date, img_date, img_dict[img_date], sw_speed))
            ### FOR DEBUGGING
            # print(f"Matched Image date {img_date} with SW date {sw_date}")
    return pd.DataFrame(matched, columns=["sw_date", "img_date", "img_file", "speed"])

# -------------------------
# Split into batches
# -------------------------
def split_batches(df, batch_size=20):
    batches, i = [], 0
    while i + batch_size <= len(df):
        window = df.iloc[i:i+batch_size]
        if (window["img_date"].diff().dropna() == pd.Timedelta(days=1)).all():
            batches.append(window)
            i += batch_size
        else:
            i += 1
    return batches

# -------------------------
# Load map file
# -------------------------
def load_map(filepath):
    with fits.open(filepath) as fit_img:
        arr = np.array(fit_img[0].data, dtype=np.float32)
    if arr.ndim == 2:
        arr = np.expand_dims(arr, -1)
    return arr



def plot_results(y_test, preds, test_batches, icme_dates, delay, fold_index,
                 threat_score, corr, p_value, save_path, max_nans=5, gap_size=3):
    """
    Plot predicted vs observed SW in a single row with compressed gaps.
    NaNs are reduced to fixed size, and gaps are represented as small spacing
    so points don't get squeezed.
    """

    # --- Prepare dataframe ---
    df_plot = pd.DataFrame({
        "sw_date": [r.sw_date for b in test_batches for _, r in b.iterrows()],
        "observed": y_test,
        "predicted": preds.flatten()
    }).sort_values("sw_date").reset_index(drop=True)

    # --- Compress long NaN runs ---
    compressed_rows = []
    nan_count = 0
    for _, row in df_plot.iterrows():
        if pd.isna(row["observed"]) or pd.isna(row["predicted"]):
            nan_count += 1
            if nan_count <= max_nans:
                compressed_rows.append(row)
        else:
            nan_count = 0
            compressed_rows.append(row)
    df_plot = pd.DataFrame(compressed_rows).reset_index(drop=True)

    # --- Build artificial index with spacing for gaps ---
    index_with_gaps = []
    current_idx = 0
    prev_date = None
    for d in df_plot["sw_date"]:
        if prev_date is not None:
            # If gap > 1 day, insert artificial spacing
            if (d - prev_date).days > 1:
                current_idx += gap_size
        index_with_gaps.append(current_idx)
        current_idx += 1
        prev_date = d
    df_plot["plot_idx"] = index_with_gaps

    idx = df_plot["plot_idx"].values
    dates = df_plot["sw_date"].values
    obs = df_plot["observed"].values
    pred = df_plot["predicted"].values

    # --- Helper: plot with gaps ---
    def plot_with_gaps(ax, x, y, color, label, gap_tolerance=gap_size, linewidth=2):
        seg_x, seg_y = [], []
        for i in range(len(x)):
            if np.isnan(y[i]):
                if seg_x:  # flush current segment
                    ax.plot(seg_x, seg_y, color=color, linewidth=linewidth, label=label if label else None)
                    label = None  # only show label once
                    seg_x, seg_y = [], []
                continue

            if seg_x and (x[i] - seg_x[-1]) > gap_tolerance:
                ax.plot(seg_x, seg_y, color=color, linewidth=linewidth, label=label if label else None)
                label = None
                seg_x, seg_y = [], []

            seg_x.append(x[i])
            seg_y.append(y[i])

        if seg_x:
            ax.plot(seg_x, seg_y, color=color, linewidth=linewidth, label=label if label else None)

    # --- Figure layout: 1 row, 2 columns ---
    fig = plt.figure(figsize=(32, 6))  # wider to stretch
    gs = GridSpec(1, 2, figure=fig, width_ratios=[8, 2], wspace=0.25)

    ax_ts = fig.add_subplot(gs[0, 0])   # time-series panel
    ax_corr = fig.add_subplot(gs[0, 1]) # correlation panel

    plt.style.use("default")
    fig.suptitle(f"SpeedNet-EUV SW Prediction VS Observed (Fold {fold_index}, Delay {delay})",
                 fontsize=16, fontweight="bold", y=1.02)

    # --- Plot with gaps properly handled ---
    plot_with_gaps(ax_ts, idx, pred, color="blue", label="PREDICTED", gap_tolerance=gap_size)
    plot_with_gaps(ax_ts, idx, obs, color="green", label="OBSERVED", gap_tolerance=gap_size)

    # ICME markers (±2 days near data points)
    for icme_date in icme_dates:
        if np.any(np.abs((dates - np.datetime64(icme_date)).astype("timedelta64[D]").astype(int)) <= 2):
            icme_idx = df_plot.loc[df_plot["sw_date"] == icme_date, "plot_idx"]
            if not icme_idx.empty:
                ax_ts.axvline(x=icme_idx.iloc[0], color="black", linestyle="-", ymax=1, ymin=0.95, linewidth=2.5)

    ax_ts.set_ylabel("V (Km/s)", fontsize=12, fontweight="bold")
    ax_ts.set_xlabel("SW DATE", fontsize=12, fontweight="bold")
    ax_ts.grid(False)

    ax_ts.text(0.5, 0.9, f"Threat Score: {threat_score:.2f}",
               transform=ax_ts.transAxes, fontsize=14, va="top", ha="center")
    ax_ts.legend(loc="upper left")

    # --- X-axis labels: only real dates at selected ticks ---
    step = max(1, len(idx)//15)  # aim ~15 ticks
    tick_locs = idx[::step]
    tick_labels = [pd.to_datetime(dates[i]).strftime("%d-%m-%Y") for i in range(0, len(dates), step)]
    ax_ts.set_xticks(tick_locs)
    ax_ts.set_xticklabels(tick_labels, rotation=45, ha="right", fontsize=8)

    # --- Correlation plot ---
    valid_mask = ~pd.isna(obs) & ~pd.isna(pred)
    ax_corr.scatter(obs[valid_mask], pred[valid_mask], alpha=0.7,
                    color="purple", label="Solar wind speed")

    slope, intercept = np.polyfit(obs[valid_mask], pred[valid_mask], 1)
    ax_corr.plot(obs[valid_mask], slope*obs[valid_mask] + intercept,
                 'k--', label="correlation")

    ax_corr.set_title("Correlation Plot", fontsize=14, fontweight="bold")
    ax_corr.set_xlabel("V(Km/s) Observed", fontsize=12, fontweight="bold")
    ax_corr.set_ylabel("V(Km/s) Predicted", fontsize=12, fontweight="bold")

    ax_corr.text(0.95, 0.05, f"r = {corr:.2f}\np = {p_value:.3e}",
                 transform=ax_corr.transAxes, fontsize=12,
                 verticalalignment="bottom", horizontalalignment="right",
                 bbox=dict(boxstyle="round", facecolor="white"))

    ax_corr.legend(loc="upper left")
    ax_corr.set_xlim(250, 750)
    ax_corr.set_ylim(250, 750)

    plt.savefig(save_path, bbox_inches="tight")
    plt.close()


# -------------------------
# Train/Val/Test splitting
# -------------------------
def split_train_val_test(df_matched, test_fraction=0.25, n_splits=5, seed=42):
    np.random.seed(seed)
    all_batches = split_batches(df_matched, batch_size=20)
    num_batches = len(all_batches)
    n_test = max(1, int(test_fraction * num_batches))
    test_indices = np.random.choice(num_batches, size=n_test, replace=False)

    test_batches = [all_batches[i] for i in test_indices]
    train_val_batches = [b for i, b in enumerate(all_batches) if i not in test_indices]

    print(f"Total batches: {num_batches}, Test batches: {len(test_batches)}, Train+Val batches: {len(train_val_batches)}")
    return train_val_batches, test_batches

# -------------------------
# Training pipeline
# -------------------------
def train_pipeline(sw_csv, icme_excel, img_path, model_path, seed=42):
    np.random.seed(seed)
    tf.random.set_seed(seed)

    df_sw, img_dict, icme_dates = prepare_dataset(sw_csv, icme_excel, img_path)
    print(f"Loaded {len(df_sw)} SW records and {len(img_dict)} images from {img_path}")

    delays = [4]  # Which is the average delay
    batchsizes = [12, 32, 64]
    lstm_units = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
    dense_units = [100, 200, 300, 400, 500]
    n_splits = 5
    full_metric_list = []
    fold_index = 0

    for delay in delays:
        if delay == 4:
            print(f"\n=== Delay {delay} ===")
            df_matched = match_data(df_sw, img_dict, delay)
            train_val_batches, test_batches = split_train_val_test(df_matched, test_fraction=0.2, n_splits=n_splits, seed=seed)

            # n_splits = 5
            num_samples = len(train_val_batches)  # number of train+val batches

            for fold_index in range(n_splits):
                val_start = fold_index * (num_samples // n_splits)
                val_end = val_start + (num_samples // n_splits)
                if val_end > num_samples:
                    val_end = num_samples

                # Define train/validation batches
                train_idx = list(range(0, val_start)) + list(range(val_end, num_samples))
                val_idx = list(range(val_start, val_end))

                # Load train/val data
                X_train = np.array([load_map(os.path.join(img_path, r.img_file))
                                    for i in train_idx for _, r in train_val_batches[i].iterrows()])
                y_train = np.array([r.speed for i in train_idx for _, r in train_val_batches[i].iterrows()])

                X_val = np.array([load_map(os.path.join(img_path, r.img_file))
                                for i in val_idx for _, r in train_val_batches[i].iterrows()])
                y_val = np.array([r.speed for i in val_idx for _, r in train_val_batches[i].iterrows()])

                # Test set remains fixed
                X_test = np.array([load_map(os.path.join(img_path, r.img_file))
                                for b in test_batches for _, r in b.iterrows()])
                y_test = np.array([r.speed for b in test_batches for _, r in b.iterrows()])

                print(f"Fold {fold_index + 1}: Train {len(X_train)}, Val {len(X_val)}, Test {len(X_test)}")


                input_shape = X_train.shape[1:]

                # Suppose we saved the last state: 
                
                # resume = {"fold": 4, "bs": 12, "lstm": 40, "dense": 400}
                # resume_found = False
                for bs in batchsizes:
                    for lstm_u in lstm_units:
                        for dense_u in dense_units:
                            # # --- Only apply resume skipping for fold ---
                            # if fold_index == resume["fold"] and not resume_found:
                            #     if (bs == resume["bs"] and 
                            #         lstm_u == resume["lstm"] and 
                            #         dense_u == resume["dense"]):
                            #         resume_found = True   # start training from here
                            #     else:
                            #         continue  # skip until resume point
                            # elif fold_index < resume["fold"]:
                            #     continue
                            
                            print(f"Training Fold {fold_index}, Delay {delay}, BS {bs}, LSTM {lstm_u}, Dense {dense_u}")
                            # model = SpeedNet(input_shape, lstm_u, dense_u)
                            # model.compile(optimizer=Adam(1e-4), loss="mse", metrics=["mae", "mse"])

                            model_filename = (f"{model_path}/best_model_fold{fold_index}_delay{delay}_"
                                            f"bs{bs}_lstm{lstm_u}_dense{dense_u}.keras")

                            # checkpoint = ModelCheckpoint(
                            #     model_filename,
                            #     monitor="val_loss",
                            #     save_best_only=True,
                            #     mode="min",
                            #     verbose=1
                            # )

                            # earlystop = EarlyStopping(
                            #     monitor="val_loss",
                            #     patience=20,
                            #     restore_best_weights=True,
                            #     verbose=1
                            # )

                            # history = model.fit(
                            #     X_train, y_train,
                            #     epochs=100,
                            #     batch_size=bs,
                            #     validation_data=(X_val, y_val),
                            #     callbacks=[checkpoint, earlystop],
                            #     verbose=1
                            # )

                            # plt.plot(history.history["loss"], label="train loss")
                            # plt.plot(history.history["val_loss"], label="val loss")
                            # plt.xlabel("Epochs"); plt.ylabel("Loss")
                            # plt.title(f"Fold {fold_index} Delay {delay} BS {bs} LSTM {lstm_u} Dense {dense_u}")
                            # plt.legend()
                            # plt.savefig(model_filename.replace(".keras","_loss.png"))
                            # plt.close()
                            

                            loaded = load_model(model_filename)
                            preds = loaded.predict(X_test, verbose=0)

                            mae = MeanAbsoluteError()(y_test, preds).numpy()
                            mse = MeanSquaredError()(y_test, preds).numpy()
                            rmse = np.sqrt(mse)
                            corr, p_value = pearsonr(tf.squeeze(y_test).numpy(), tf.squeeze(preds).numpy())

                            # fisher transform / uncertainty
                            n = max(1, len(np.squeeze(y_test)))
                            z = 0.5 * np.log((1 + corr) / (1 - corr + 1e-12) + 0.0)
                            se = 1.0 / np.sqrt(max(1, n - 3))
                            z_conf_interval = se * 1.96
                            pearson_uncertainty = np.tanh(z_conf_interval)
            
                            mse_uncertainty = float(np.sqrt(2 * (mse ** 2) / max(1, n)))
                            rmse_uncertainty = 0.5 * mse_uncertainty / (rmse + 1e-12)
            
                            # threat score via hsemetric if available (BatchwiseHSE expects arrays)
                            if hsemetric is not None:
                                try:
                                    tp, fp, fn = hsemetric.BatchwiseHSE(y_test, np.array(preds))
                                    threat_score = tp * 1.0 / (tp + fp + fn) if (tp + fp + fn) > 0 else np.nan
                                except Exception:
                                    threat_score = np.nan
                            else:
                                threat_score = np.nan
            

                            row = (fold_index, delay, bs, lstm_u, dense_u, mae, mse, rmse, corr, p_value, threat_score)
                            full_metric_list.append(row)

                            plot_results(
                                y_test, preds, test_batches, icme_dates,
                                delay, fold_index, threat_score=threat_score, corr=corr, p_value=p_value,
                                save_path=model_filename.replace(".keras", "_results.png")
                            )

    df_metrics = pd.DataFrame(full_metric_list,
                              columns=["Fold","Delay","Batchsize","LSTM","Dense","MAE","MSE","RMSE","Corr","P_Value", "Threat_Score"])
    df_metrics.to_excel(f"{model_path}/three_channel_ablation_full_metrics.xlsx", index=False)
    print("\nTraining complete. Metrics saved.")




### RUN THE ABLATION FOR THE THREE CHANNEL MODEL
`SPEEDNET-EUV`

In [ ]:
train_pipeline(
    "../DATASET/Omni_velocity.csv",
    "../DATASET/Original_Full_Richardson_cane_list(2010-2023).xlsx",
    "../DATASET/stonyhurst_three_channel/",
    "../DATASET/Results/Ablation/save_model_channel"
)

Loaded 4510 SW records and 3427 images from /home/joseph/DL/Revised_SpeedNet/DATASET/stonyhurst_three_channel/

=== Delay 4 ===
Total batches: 75, Test batches: 15, Train+Val batches: 60
Fold 1: Train 960, Val 240, Test 300
Training Fold 0, Delay 4, BS 12, LSTM 10, Dense 100


2025-09-02 04:36:37.164508: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-09-02 04:36:37.905647: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:961] could not open file to read NUMA node: /sys/bus/pci/devices/0000:17:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-09-02 04:36:37.905736: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:961] could not open file to read NUMA node: /sys/bus/pci/devices/0000:73:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-09-02 04:36:37.905788: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:961] could not open file to read NUMA node: /sys/bus/pci/devices/0000:17:00.0/numa_node
Your kernel may h

Training Fold 0, Delay 4, BS 12, LSTM 10, Dense 200
Training Fold 0, Delay 4, BS 12, LSTM 10, Dense 300
Training Fold 0, Delay 4, BS 12, LSTM 10, Dense 400
Training Fold 0, Delay 4, BS 12, LSTM 10, Dense 500
Training Fold 0, Delay 4, BS 12, LSTM 20, Dense 100
Training Fold 0, Delay 4, BS 12, LSTM 20, Dense 200
Training Fold 0, Delay 4, BS 12, LSTM 20, Dense 300
Training Fold 0, Delay 4, BS 12, LSTM 20, Dense 400
Training Fold 0, Delay 4, BS 12, LSTM 20, Dense 500
Training Fold 0, Delay 4, BS 12, LSTM 30, Dense 100
Training Fold 0, Delay 4, BS 12, LSTM 30, Dense 200
Training Fold 0, Delay 4, BS 12, LSTM 30, Dense 300
Training Fold 0, Delay 4, BS 12, LSTM 30, Dense 400
Training Fold 0, Delay 4, BS 12, LSTM 30, Dense 500
Training Fold 0, Delay 4, BS 12, LSTM 40, Dense 100
Training Fold 0, Delay 4, BS 12, LSTM 40, Dense 200
Training Fold 0, Delay 4, BS 12, LSTM 40, Dense 300
Training Fold 0, Delay 4, BS 12, LSTM 40, Dense 400
Training Fold 0, Delay 4, BS 12, LSTM 40, Dense 500
Training Fol

### RUN THE ABLATION FOR THE BINARY MODEL
`SPEEDNET-BM`

In [ ]:
train_pipeline(
    "../DATASET/Omni_velocity.csv",
    "../DATASET/Original_Full_Richardson_cane_list(2010-2023).xlsx",
    "../DATASET/binary_map/",
    "../DATASET/Results/Ablation/save_model_binary"
)